In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import numpy as np 
import sympy as sp
from sympy import init_printing
init_printing()

import fittingError

# General Variables

In [ ]:
# Define symbolic variable
lambda_ = sp.symbols('lambda', positive=True)

# Deformation gradient
F = sp.Matrix([
    [lambda_, 0, 0],
    [0, 1/sp.sqrt(lambda_), 0],
    [0, 0, 1/sp.sqrt(lambda_)]
])

In [ ]:
# Right Cauchy-Green deformation tensor
C = F.T * F

# Invariants
I_1 = sp.trace(C)
I_2 = sp.Rational(1, 2) * (sp.trace(C)**2 - sp.trace(C * C))

# Mooney Rivlin Constants 
C_10 = sp.symbols('C_10')
C_01 = sp.symbols('C_01')
C_11 = sp.symbols('C_11')
C_20 = sp.symbols('C_20')
C_02 = sp.symbols('C_02')

# Ogden Constants
mu_1 = sp.symbols('mu_1')
mu_2 = sp.symbols('mu_2')
mu_3 = sp.symbols('mu_3')
alpha_1 = sp.symbols('alpha_1')
alpha_2 = sp.symbols('alpha_2')
alpha_3 = sp.symbols('alpha_3')


# Material Models

## Mooney Rivlin N=2

In [ ]:
W_MR2 = (
    C_10 * (I_1 - 3)
    + C_01 * (I_2 - 3)
    + C_11 * (I_1 - 3) * (I_2 - 3)
    + C_20 * (I_1 - 3)**2
    + C_02 * (I_2 - 3)**2
)
P11_MR2 = sp.simplify(sp.diff(W_MR2, lambda_))
MR2_Params = [lambda_, C_10, C_01, C_11, C_20, C_02]
stressStrainFunction_MR2 = sp.lambdify(MR2_Params, P11_MR2)

# Mooney Rivlin N=1

In [ ]:
W_MR1 = (
    C_10 * (I_1 - 3)
    + C_01 * (I_2 - 3)
)
P11_MR1 = sp.simplify(sp.diff(W_MR1, lambda_))
MR1_Params = [lambda_, C_10, C_01]
stressStrainFunction_MR1 = sp.lambdify(MR1_Params, P11_MR1)

## Simplified Mooney Rivlin

In [ ]:
W_MR_simplified = (
    C_10 * (I_1 - 3)
    + C_01 * (I_2 - 3)
    + C_02 * (I_2 - 3)**2
)
P11_MR_simplified = sp.simplify(sp.diff(W_MR_simplified, lambda_))
MR_simplified_Params = [lambda_, C_10, C_01, C_02]
stressStrainFunction_MR_simplified = sp.lambdify(MR_simplified_Params, P11_MR_simplified)

# Trinomial Model

In [ ]:
W_trinomial = (
    C_10 * (I_1 - 3)
    + C_01 * (I_2 - 3)
    + C_20 * (I_1 - 3)**2
)
P11_trinomial = sp.simplify(sp.diff(W_trinomial, lambda_))
trinomial_Params = [lambda_, C_10, C_01, C_20]
stressStrainFunction_trinomial = sp.lambdify(trinomial_Params, P11_trinomial)

# Reduced Polynomial Model N=2

In [ ]:
W_reducedPolynomial = (
    C_10 * (I_1 - 3)
    + C_20 * (I_1 - 3)**2
)
P11_reducedPolynomial = sp.simplify(sp.diff(W_reducedPolynomial, lambda_))
reducedPolynomial_Params = [lambda_, C_10, C_20]
stressStrainFunction_reducedPolynomial = sp.lambdify(reducedPolynomial_Params, P11_reducedPolynomial)

# Neo Hook

In [ ]:
W_neoHook = (
    C_10 * (I_1 - 3)
)
P11_neoHook = sp.simplify(sp.diff(W_neoHook, lambda_))
neoHook_Params = [lambda_, C_10]
stressStrainFunction_neoHook = sp.lambdify(neoHook_Params, P11_neoHook)

# Ogden N=1

In [ ]:
W_ogdenN1 = (
    mu_1/alpha_1*(lambda_**alpha_1 + (1/sp.sqrt(lambda_))**alpha_1 + (1/sp.sqrt(lambda_))**alpha_1)
)
P11_ogdenN1 = sp.simplify(sp.diff(W_ogdenN1, lambda_))
ogdenN1_Params = [lambda_, mu_1, alpha_1]
stressStrainFunction_ogdenN1 = sp.lambdify(ogdenN1_Params, P11_ogdenN1)


In [ ]:
W_ogdenN2 = (
    mu_1/alpha_1*(lambda_**alpha_1 + (1/sp.sqrt(lambda_))**alpha_1 + (1/sp.sqrt(lambda_))**alpha_1) + 
    mu_2/alpha_2*(lambda_**alpha_2 + (1/sp.sqrt(lambda_))**alpha_2 + (1/sp.sqrt(lambda_))**alpha_2)
)
P11_ogdenN2 = sp.simplify(sp.diff(W_ogdenN2, lambda_))
ogdenN2_Params = [lambda_, mu_1, mu_2, alpha_1, alpha_2]
stressStrainFunction_ogdenN2 = sp.lambdify(ogdenN2_Params, P11_ogdenN2)

In [ ]:
W_ogdenN3 = (
    mu_1/alpha_1*(lambda_**alpha_1 + (1/sp.sqrt(lambda_))**alpha_1 + (1/sp.sqrt(lambda_))**alpha_1) + 
    mu_2/alpha_2*(lambda_**alpha_2 + (1/sp.sqrt(lambda_))**alpha_2 + (1/sp.sqrt(lambda_))**alpha_2) + 
    mu_3/alpha_3*(lambda_**alpha_3 + (1/sp.sqrt(lambda_))**alpha_3 + (1/sp.sqrt(lambda_))**alpha_3)
)
P11_ogdenN3 = sp.simplify(sp.diff(W_ogdenN3, lambda_))
ogdenN3_Params = [lambda_, mu_1, mu_2, mu_3, alpha_1, alpha_2, alpha_3]
stressStrainFunction_ogdenN3 = sp.lambdify(ogdenN3_Params, P11_ogdenN3)

# Curve Fitting 

In [ ]:
def fitListOfFunctionsToGroundTruth(listOfFunctions, listOfFunctionNames, listOfParams, groundTruthDf): 
    xGroundTruth = groundTruthDf["nominalStrain"] + 1
    yGroundTruth = groundTruthDf["nominalStressInMPa"]

    fig, ax = plt.subplots(1,1, figsize=(12,5))
    plt.plot(xGroundTruth-1, yGroundTruth, label="Ground Truth", linewidth=3)
    for i, f in enumerate(listOfFunctions): 
        popt, pcov = curve_fit(f, xGroundTruth, yGroundTruth, maxfev=100000)
        labelstring = f"{listOfFunctionNames[i]}: "
        for j, v in enumerate(popt):
            labelstring += f"{listOfParams[i][j+1]}: {round(popt[j],2)}  "
        yFitted = f(xGroundTruth, *popt)
        dfFitted = pd.DataFrame(data={"nominalStrain":xGroundTruth-1, "nominalStressInMPa":yFitted}) # do minus one again
        err = fittingError.fittingError(groundTruthDf, dfFitted)
        labelstring += f". Fitting error: {round(err, 3)}"
        plt.plot(xGroundTruth-1, yFitted, label=labelstring, linewidth=2, linestyle="dashed")
    plt.legend()
    plt.xlabel("Nominal Strain")
    plt.ylabel("Nominal Stress [MPa]")
    plt.show()

In [ ]:
import pandas as pd 
tpu1GroundTruth = pd.read_csv("TPU-test-data/stressStrainCurveTPU1.csv")
tpu2GroundTruth = pd.read_csv("TPU-test-data/stressStrainCurveTPU2.csv")
tpu3GroundTruth = pd.read_csv("TPU-test-data/stressStrainCurveTPU3.csv")


In [ ]:
listOfFunctions = [
    stressStrainFunction_MR1, 
    stressStrainFunction_MR2, 
    stressStrainFunction_MR_simplified, 
    stressStrainFunction_trinomial, 
    stressStrainFunction_reducedPolynomial, 
    stressStrainFunction_neoHook, 
    # stressStrainFunction_ogdenN1, 
    # stressStrainFunction_ogdenN2, 
    # stressStrainFunction_ogdenN3, 
    ]

listOfFunctionNames = [
    "MR1", 
    "MR2", 
    "MR_simplified", 
    "Trinomial", 
    "Reduced Polynomial", 
    "Neo Hook", 
    # "Ogden1", 
    # "Ogden2", 
    # "Ogden3", 
    ]

listOfParams = [
    MR1_Params, 
    MR2_Params, 
    MR_simplified_Params, 
    trinomial_Params, 
    reducedPolynomial_Params, 
    neoHook_Params, 
    # ogdenN1_Params, 
    # ogdenN2_Params, 
    # ogdenN3_Params, 
    ]



In [ ]:
# plot just ground truth
fitListOfFunctionsToGroundTruth([], [], [], tpu1GroundTruth)

In [ ]:
fitListOfFunctionsToGroundTruth(listOfFunctions, listOfFunctionNames, listOfParams, tpu1GroundTruth)

In [ ]:
# Mooney-Rivlin simplified Material parameters for TPU1 from Wang paper: 
# C_10 = -11.11   
# C_01 = 17.4
# C_02 = 3.134


In [ ]:
fitListOfFunctionsToGroundTruth(listOfFunctions, listOfFunctionNames, listOfParams, tpu2GroundTruth)

In [ ]:
fitListOfFunctionsToGroundTruth(listOfFunctions, listOfFunctionNames, listOfParams, tpu3GroundTruth)